# Experiment: Dynamic Rule Selector Demo


This notebook demos a boundary-token update loop for dynamic rule selection:

1. Load one RuleArena airline sample.
2. Get applicable rules and oracle-derived execution order.
3. Generate the answer one sentence at a time.
4. After each sentence, call an external evaluator LLM to choose which rule(s) to boost next.

For now, this notebook does selection only (no attention boosting intervention).


In [1]:
from pathlib import Path
import importlib.util
import types
import sys
import json
import os
import re
from pprint import pprint


def load_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

print("Project root:", ROOT)

seg_mod = load_module("src.rulearena.rulebook_segments", ROOT / "src/rulearena/rulebook_segments.py")
app_mod = load_module("src.rulearena.rule_applicability", ROOT / "src/rulearena/rule_applicability.py")

src_pkg = types.ModuleType("src")
src_pkg.__path__ = [str(ROOT / "src")]
sys.modules["src"] = src_pkg

rulearena_pkg = types.ModuleType("src.rulearena")
rulearena_pkg.__path__ = [str(ROOT / "src/rulearena")]
sys.modules["src.rulearena"] = rulearena_pkg

sys.modules["src.rulearena.rulebook_segments"] = seg_mod
sys.modules["src.rulearena.rule_applicability"] = app_mod

oracle_mod = load_module("src.rulearena.rule_application_oracle", ROOT / "src/rulearena/rule_application_oracle.py")

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except Exception:
    OPENAI_AVAILABLE = False


Project root: /Users/vitoriag/Documents/multi-rules


## Sample Selection


In [2]:
COMP = 0
SAMPLE_IDX = 1

rulebook_path = ROOT / "datasets/RuleArena/airline/reference_rules.txt"
problems_path = ROOT / f"datasets/RuleArena/airline/synthesized_problems/comp_{COMP}.jsonl"

with open(problems_path) as f:
    for i, line in enumerate(f):
        if i == SAMPLE_IDX:
            sample = json.loads(line)
            break
    else:
        raise IndexError(f"Sample {SAMPLE_IDX} not found in {problems_path}")

question_prompt = sample["prompt"]
info = sample["info"]

print(f"Loaded comp_{COMP} sample {SAMPLE_IDX}")
print("\nQuestion prompt:")
print(question_prompt)
print("\nInfo dict:")
pprint(info)


Loaded comp_0 sample 1

Question prompt:
Linda is a Business Class passenger flying from Charlotte to Phoenix with the following items:
1. A backpack: 18 x 13 x 6 inches, 8 lbs;
2. A luggage box: 41 x 20 x 16 inches, 95 lbs;
3. A backpack: 38 x 24 x 18 inches, 74 lbs;
4. A backpack: 37 x 16 x 10 inches, 54 lbs;
5. A backpack: 43 x 25 x 20 inches, 52 lbs;

Linda's flight ticket is $186.

Info dict:
{'bag_list': [{'id': 1, 'name': 'backpack', 'size': [18, 13, 6], 'weight': 8},
              {'id': 2,
               'name': 'luggage box',
               'size': [41, 20, 16],
               'weight': 95},
              {'id': 3, 'name': 'backpack', 'size': [38, 24, 18], 'weight': 74},
              {'id': 4, 'name': 'backpack', 'size': [37, 16, 10], 'weight': 54},
              {'id': 5,
               'name': 'backpack',
               'size': [43, 25, 20],
               'weight': 52}],
 'base_price': 186,
 'customer_class': 'Business',
 'direction': 1,
 'routine': 'U.S.'}


## Build Rule Set And Oracle Order


In [3]:
rulebook_text = rulebook_path.read_text()
coarse_segments = seg_mod.get_coarse_segments(rulebook_text)
fine_segments = seg_mod.get_fine_segments(rulebook_text)

applied = app_mod.get_applied_rules_with_coarse(info, fine_segments, coarse_segments)
trace = oracle_mod.get_rule_application_trace(info, fine_segments)

ordered_unique_rules = []
seen = set()
for step in trace["steps"]:
    name = step["rule_name"]
    if name not in seen:
        seen.add(name)
        ordered_unique_rules.append(name)

relevant_fine_rules = [seg["name"] for seg in applied["fine"]]

print("Applicable coarse sections:", len(applied["coarse"]))
print([seg["name"] for seg in applied["coarse"]])

print("\nApplicable fine rules:", len(relevant_fine_rules))
print(relevant_fine_rules)

print("\nOracle ordered unique rules:", len(ordered_unique_rules))
for i, r in enumerate(ordered_unique_rules):
    print(f"{i:2d}. {r}")


Applicable coarse sections: 9
['preamble', 'carry_on', 'checked_bags_intro', 'first_bag', 'second_bag', 'third_bag', 'fourth_bag', 'complimentary_bags', 'weight_and_size']

Applicable fine rules: 29
['preamble/all_published_bag_fees', 'carry_on/you_re_allowed_1', 'carry_on/your_personal_item_like', 'carry_on/these_don_t_count', 'carry_on/you_can_bring_only', 'carry_on/the_total_size_of', 'carry_on/your_soft_sided_garment', 'checked_bags_intro/bag_fees_have_been', 'checked_bags_intro/travel_within_between_the', 'checked_bags_intro/travel_to_from_canada', 'checked_bags_intro/all_bag_fees_are', 'first_bag/row_us_puerto_rico', 'second_bag/row_us_canada_puerto', 'third_bag/row_us_canada_puerto', 'fourth_bag/row_us_canada_puerto', 'complimentary_bags/in_some_cases_you', 'complimentary_bags/if_your_status_level', 'complimentary_bags/free_checked_bags_may', 'complimentary_bags/1st_checked_bag_is', 'complimentary_bags/or_when_traveling_to', 'complimentary_bags/1st_and_2nd_checked', 'complimenta

## Generator And Selector Setup

Configure models:
- `GEN_MODEL`: sentence-by-sentence generator
- `SELECTOR_MODEL`: evaluator that picks rules to boost next

If OpenAI is unavailable, selector falls back to a simple keyword heuristic.


In [4]:
GEN_MODEL = "gpt-4o-mini"
SELECTOR_MODEL = "gpt-4o-mini"

MAX_STEPS = 12
START_RULE_PTR = 0

USE_FILTERED_RULEBOOK = True
if USE_FILTERED_RULEBOOK:
    rules_for_prompt = app_mod.build_filtered_rulebook(info, rulebook_text, fine_segments, coarse_segments)
else:
    rules_for_prompt = rulebook_text

if OPENAI_AVAILABLE and os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
else:
    client = None

print("OpenAI available:", OPENAI_AVAILABLE)
print("API key present:", bool(os.environ.get("OPENAI_API_KEY")))


OpenAI available: False
API key present: False


In [5]:
def generate_next_sentence(current_generation: str) -> str:
    # Generate one incremental sentence (boundary-based step)
    if client is None:
        fallback = [
            "First, I identify which items are checked bags and which are carry-on.",
            "Next, I determine the base fee for the first checked bag based on route and cabin.",
            "Then I evaluate overweight and oversize penalties for that bag.",
            "I apply the higher of overweight and oversize fee where both apply.",
            "I repeat the same process for the remaining checked bags.",
            "Finally, I sum ticket price and all bag-related fees.",
            "The total cost is $0.",
        ]
        idx = min(len(re.findall(r"\.", current_generation)), len(fallback) - 1)
        return fallback[idx]

    system = "You are a careful airline fee assistant. Continue the solution by exactly ONE sentence."
    user = (
        "You are solving an airline fee problem. Continue from the current partial solution with one next sentence only. "
        "Do not restart from scratch.\n\n"
        f"Problem:\n{question_prompt}\n\n"
        f"Relevant Rules:\n{rules_for_prompt}\n\n"
        f"Current solution so far:\n{current_generation}\n"
    )

    resp = client.chat.completions.create(
        model=GEN_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    text = resp.choices[0].message.content.strip()

    if not re.search(r"[.!?]$", text):
        text += "."
    return text


def heuristic_selector(current_generation: str, current_ptr: int) -> dict:
    # Simple fallback selector when API is unavailable
    lower = current_generation.lower()
    advance_cues = ["next", "then", "second", "third", "finally"]
    should_advance = any(cue in lower[-220:] for cue in advance_cues)

    if should_advance and current_ptr < len(ordered_unique_rules) - 1:
        new_ptr = current_ptr + 1
        selected = [ordered_unique_rules[new_ptr]]
        decision = "advance"
        conf = 0.58
    else:
        new_ptr = current_ptr
        selected = [ordered_unique_rules[current_ptr]]
        decision = "stay"
        conf = 0.45

    return {
        "decision": decision,
        "selected_rules": selected,
        "next_rule_pointer": new_ptr,
        "confidence": conf,
        "reason": "heuristic fallback",
    }


def external_selector(current_generation: str, current_ptr: int) -> dict:
    # LLM selector: choose rules to boost next (no weights)
    if client is None:
        return heuristic_selector(current_generation, current_ptr)

    rules_context = [{"idx": i, "name": name} for i, name in enumerate(ordered_unique_rules)]

    selector_prompt = {
        "task": "Select which rule(s) should be boosted for the NEXT generation chunk.",
        "constraints": {
            "max_selected_rules": 3,
            "valid_decisions": ["stay", "advance", "jump"],
            "pointer_bounds": [0, len(ordered_unique_rules) - 1],
        },
        "current_rule_pointer": current_ptr,
        "ordered_rules": rules_context,
        "current_generation": current_generation,
        "expected_output_json": {
            "decision": "stay|advance|jump",
            "next_rule_pointer": "int",
            "selected_rule_indices": ["int"],
            "confidence": "float in [0,1]",
            "reason": "short string",
        },
    }

    system = "You are a strict JSON controller for rule selection. Output JSON only."
    user = json.dumps(selector_prompt, indent=2)

    resp = client.chat.completions.create(
        model=SELECTOR_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )

    raw = resp.choices[0].message.content.strip()
    try:
        out = json.loads(raw)
    except Exception:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            return heuristic_selector(current_generation, current_ptr)
        out = json.loads(match.group(0))

    ptr = int(out.get("next_rule_pointer", current_ptr))
    ptr = max(0, min(ptr, len(ordered_unique_rules) - 1))

    idxs = out.get("selected_rule_indices", [])
    if not isinstance(idxs, list):
        idxs = [ptr]
    idxs = [int(i) for i in idxs if isinstance(i, int) or (isinstance(i, str) and i.isdigit())]
    idxs = [i for i in idxs if 0 <= i < len(ordered_unique_rules)]
    if len(idxs) == 0:
        idxs = [ptr]
    idxs = idxs[:3]

    confidence = out.get("confidence", 0.0)
    try:
        confidence = float(confidence)
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    return {
        "decision": str(out.get("decision", "stay")),
        "selected_rules": [ordered_unique_rules[i] for i in idxs],
        "selected_rule_indices": idxs,
        "next_rule_pointer": ptr,
        "confidence": confidence,
        "reason": str(out.get("reason", "")),
        "raw": raw,
    }


## Run Stepwise Generation And Rule Selection


In [6]:
history = []
current_generation = ""
rule_ptr = START_RULE_PTR

for step in range(1, MAX_STEPS + 1):
    sentence = generate_next_sentence(current_generation)
    current_generation = (current_generation + " " + sentence).strip()

    selection = external_selector(current_generation, rule_ptr)

    if selection["confidence"] < 0.35:
        selection = {
            "decision": "stay",
            "selected_rules": [ordered_unique_rules[rule_ptr]],
            "selected_rule_indices": [rule_ptr],
            "next_rule_pointer": rule_ptr,
            "confidence": selection["confidence"],
            "reason": "low confidence fallback",
        }

    rule_ptr = selection["next_rule_pointer"]

    row = {
        "step": step,
        "sentence": sentence,
        "decision": selection["decision"],
        "next_rule_pointer": selection["next_rule_pointer"],
        "selected_rules": selection["selected_rules"],
        "confidence": selection["confidence"],
        "reason": selection.get("reason", ""),
    }
    history.append(row)

    print(f"\n--- Step {step} ---")
    print("Sentence:", sentence)
    print(
        "Selection:",
        {
            "decision": row["decision"],
            "next_rule_pointer": row["next_rule_pointer"],
            "selected_rules": row["selected_rules"],
            "confidence": row["confidence"],
        },
    )

    if "the total cost is" in current_generation.lower():
        break



--- Step 1 ---
Sentence: First, I identify which items are checked bags and which are carry-on.
Selection: {'decision': 'stay', 'next_rule_pointer': 0, 'selected_rules': ['preamble/all_published_bag_fees'], 'confidence': 0.45}

--- Step 2 ---
Sentence: Next, I determine the base fee for the first checked bag based on route and cabin.
Selection: {'decision': 'advance', 'next_rule_pointer': 1, 'selected_rules': ['carry_on/you_re_allowed_1'], 'confidence': 0.58}

--- Step 3 ---
Sentence: Then I evaluate overweight and oversize penalties for that bag.
Selection: {'decision': 'advance', 'next_rule_pointer': 2, 'selected_rules': ['carry_on/your_personal_item_like'], 'confidence': 0.58}

--- Step 4 ---
Sentence: I apply the higher of overweight and oversize fee where both apply.
Selection: {'decision': 'advance', 'next_rule_pointer': 3, 'selected_rules': ['carry_on/these_don_t_count'], 'confidence': 0.58}

--- Step 5 ---
Sentence: I repeat the same process for the remaining checked bags.
Sel

## Inspect Outputs


In [7]:
print("Final generation:\n")
print(current_generation)

print("\n\nSelection history:\n")
for row in history:
    print(json.dumps(row, indent=2))


Final generation:

First, I identify which items are checked bags and which are carry-on. Next, I determine the base fee for the first checked bag based on route and cabin. Then I evaluate overweight and oversize penalties for that bag. I apply the higher of overweight and oversize fee where both apply. I repeat the same process for the remaining checked bags. Finally, I sum ticket price and all bag-related fees. The total cost is $0.


Selection history:

{
  "step": 1,
  "sentence": "First, I identify which items are checked bags and which are carry-on.",
  "decision": "stay",
  "next_rule_pointer": 0,
  "selected_rules": [
    "preamble/all_published_bag_fees"
  ],
  "confidence": 0.45,
  "reason": "heuristic fallback"
}
{
  "step": 2,
  "sentence": "Next, I determine the base fee for the first checked bag based on route and cabin.",
  "decision": "advance",
  "next_rule_pointer": 1,
  "selected_rules": [
    "carry_on/you_re_allowed_1"
  ],
  "confidence": 0.58,
  "reason": "heuris

## Notes

- This demo updates selector decisions at sentence boundaries.
- No boosting is applied yet; this is only to validate selector behavior.
- Next step: wire `selected_rules` into your attention-bias mask update at each boundary.
